[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/Applied-Scientist-Interview-Gauntlet/blob/main/tutorial/04_deep_learning_transformers/deep_learning_lab.ipynb)

# Deep Learning Lab — Gradients, Normalization, Attention Cost

| | |
|---|---|
| **Companion to** | Session 4 — Deep Learning & Transformers |
| **Runtime** | CPU only (NumPy + tiny PyTorch tensors; no model downloads) |
| **Estimated time** | 30 minutes |
| **Last verified** | 2026-07-29 |

Session 4 asks you to *derive* things. This notebook makes each derivation something you have **checked**, which is what lets you state it under pressure without hedging.

You will:

1. **Verify your hand-derived backprop against autograd** — if your algebra is right, the numbers match to floating point.
2. **Watch gradients vanish** across depth, and watch a residual connection rescue them.
3. **Confirm what LayerNorm computes** — and that it is genuinely batch-independent while BatchNorm is not.
4. **Measure the O(n²) wall** in attention, and the **KV cache** growth that forces GQA.
5. **Reproduce an fp16 overflow** and watch loss scaling fix it.

> **Nothing here goes in the Metric Vault.** These are demonstrations of general mechanisms, not measurements of your projects.

In [ ]:
%pip install -q "numpy>=1.26" "torch>=2.2" "matplotlib>=3.8"

In [ ]:
import numpy as np, torch, torch.nn as nn, math
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
torch.set_printoptions(precision=6, sci_mode=False)
plt.rcParams["figure.figsize"] = (10, 4); plt.rcParams["axes.grid"] = True; plt.rcParams["grid.alpha"] = .3
print("torch", torch.__version__)

---
## Part 1 — Check your backprop derivation against autograd

Lesson 1 claims that for $z = Wx + b$, $a = \sigma(z)$:

$$\delta_z = \delta_a \odot \sigma'(z), \qquad \frac{\partial L}{\partial W} = \delta_z x^\top, \qquad \frac{\partial L}{\partial b} = \delta_z, \qquad \frac{\partial L}{\partial x} = W^\top \delta_z$$

That is a claim you can *test*. Below we compute the gradients by hand and compare against PyTorch's autograd. Agreement to ~1e-7 means the algebra is right — and now you can assert it in an interview without hedging.

In [ ]:
D_in, D_out = 5, 3
W = torch.randn(D_out, D_in, dtype=torch.float64, requires_grad=True)
b = torch.randn(D_out,       dtype=torch.float64, requires_grad=True)
x = torch.randn(D_in,        dtype=torch.float64, requires_grad=True)

# ---- forward (autograd) ----
z = W @ x + b
a = torch.tanh(z)
L = (a ** 2).sum()          # any scalar loss; dL/da = 2a
L.backward()

# ---- the same gradients, by hand ----
z_ = (W @ x + b).detach()
delta_a = 2 * torch.tanh(z_)              # dL/da for L = sum(a^2)
delta_z = delta_a * (1 - torch.tanh(z_) ** 2)   # tanh'(z) = 1 - tanh(z)^2
grad_W  = torch.outer(delta_z, x.detach())      # outer product -> shape of W
grad_b  = delta_z
grad_x  = W.detach().T @ delta_z

print(f"{'quantity':<10} {'max |autograd - hand|':>24}")
for name, auto, hand in [("dL/dW", W.grad, grad_W), ("dL/db", b.grad, grad_b), ("dL/dx", x.grad, grad_x)]:
    print(f"{name:<10} {(auto - hand).abs().max().item():>24.2e}")

print(f"\nshapes  W {tuple(W.shape)}  dL/dW {tuple(grad_W.shape)}   <- outer product matches W")
print("all match:", all(torch.allclose(a_, h_) for a_, h_ in
                        [(W.grad, grad_W), (b.grad, grad_b), (x.grad, grad_x)]))

**They agree to machine precision.** Three things this makes concrete:

- $\partial L/\partial W$ is an **outer product** $\delta_z x^\top$, which is why it has exactly $W$'s shape.
- The activation contributes an **elementwise** factor (its Jacobian is diagonal), not a matrix multiply.
- $\partial L/\partial x = W^\top \delta_z$ — **the same weights carry signal forward and gradient backward.**

And note what the hand computation needed: `x` itself. That is the concrete reason forward activations must be stored until the backward pass arrives — the activation-memory term in your Session 1 budget.

---
## Part 2 — Watch gradients vanish, then watch residuals rescue them

Lesson 1's claim: the gradient at layer $k$ is a **product of Jacobians**, so it is exponential in depth. With a saturating activation the product decays geometrically; with a residual connection the Jacobian becomes $I + \partial F/\partial h$ and an undiminished path always exists.

Below: two 40-layer stacks, identical except for the skip connection. We measure the gradient norm arriving at each layer.

In [ ]:
class DeepStack(nn.Module):
    """40 layers of Linear + activation, optionally with residual connections."""
    def __init__(self, depth=40, width=64, residual=False, act=nn.Tanh):
        super().__init__()
        self.residual = residual
        self.layers = nn.ModuleList([nn.Linear(width, width) for _ in range(depth)])
        self.acts   = nn.ModuleList([act() for _ in range(depth)])
        # Deliberately modest init so the vanishing effect is visible.
        for lin in self.layers:
            nn.init.normal_(lin.weight, std=0.5 / math.sqrt(width))
            nn.init.zeros_(lin.bias)

    def forward(self, h):
        self.hidden = []
        for lin, act in zip(self.layers, self.acts):
            h.retain_grad(); self.hidden.append(h)
            out = act(lin(h))
            h = h + out if self.residual else out
        return h

def grad_norms(residual):
    torch.manual_seed(0)
    model = DeepStack(residual=residual)
    x = torch.randn(16, 64, requires_grad=True)
    model(x).pow(2).sum().backward()
    return np.array([h.grad.norm().item() for h in model.hidden])

plain = grad_norms(residual=False)
resid = grad_norms(residual=True)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogy(plain, "o-", ms=3, label="plain stack", color="tab:red")
ax.semilogy(resid, "s-", ms=3, label="with residual connections", color="tab:blue")
ax.set_xlabel("layer index (0 = closest to the input)")
ax.set_ylabel("gradient norm reaching this layer (log scale)")
ax.set_title("Gradient flow across 40 layers"); ax.legend()
plt.tight_layout(); plt.show()

print(f"plain    : layer 39 {plain[-1]:.3e}   layer 0 {plain[0]:.3e}   ratio {plain[0]/plain[-1]:.2e}")
print(f"residual : layer 39 {resid[-1]:.3e}   layer 0 {resid[0]:.3e}   ratio {resid[0]/resid[-1]:.2e}")

**The plain stack's gradient decays by orders of magnitude** on the way down — the early layers receive almost nothing, so they barely learn. That is vanishing gradients, measured rather than asserted.

**The residual stack holds its gradient** across the full depth. The mechanism is exactly the derivative in Lesson 1: with $h_{out} = h + F(h)$, the Jacobian is $I + \partial F/\partial h$, so there is always an identity route by which gradient arrives undiminished. Vanishing would require *every* path to decay, and the skip provides one that cannot.

This single plot is why 100-layer networks became trainable.

---
## Part 3 — What LayerNorm computes, and why BatchNorm can't replace it

Lesson 2's claim: **the axis is the whole answer.** LayerNorm takes statistics across the *features of one token*; BatchNorm takes them across the *batch* per feature. Consequence: LayerNorm is batch-independent, BatchNorm is not.

Test it directly — run the same example alone, then inside a batch, and see whether the output changes.

In [ ]:
D = 8
x_single = torch.randn(1, D)
x_batch  = torch.cat([x_single, torch.randn(7, D) * 5 + 3])   # very different companions

ln = nn.LayerNorm(D); bn = nn.BatchNorm1d(D)
ln.eval(); bn.train()          # BatchNorm in train mode uses batch statistics

ln_alone, ln_in_batch = ln(x_single)[0], ln(x_batch)[0]

print("LayerNorm - same token, alone vs inside a batch of 8:")
print(f"  max |difference| = {(ln_alone - ln_in_batch).abs().max().item():.2e}  -> identical\n")

print("Manual check that LayerNorm normalises across FEATURES of one token:")
mu, var = x_single.mean(), x_single.var(unbiased=False)
manual = (x_single - mu) / torch.sqrt(var + ln.eps)
print(f"  max |nn.LayerNorm - manual| = {(ln(x_single) - manual).abs().max().item():.2e}")
print(f"  normalised token: mean {manual.mean().item():+.2e}  std {manual.std(unbiased=False).item():.4f}\n")

bn_out_batch = bn(x_batch)[0]
bn2 = nn.BatchNorm1d(D); bn2.train()
other_batch = torch.cat([x_single, torch.randn(7, D) * 0.1])
print("BatchNorm - same token, inside two different batches:")
print(f"  max |difference| = {(bn_out_batch - bn2(other_batch)[0]).abs().max().item():.3f}"
      "  -> the SAME token gets a DIFFERENT output")

In [ ]:
# What happens to BatchNorm with a batch of one -- the autoregressive-decoding case.
bn3 = nn.BatchNorm1d(D); bn3.train()
try:
    out = bn3(x_single)
    print("batch size 1 in train mode -> output:", out.detach().numpy().round(3))
    print("variance over a single example is 0, so the normalised output collapses.")
except Exception as e:
    print("batch size 1 in train mode raises:", type(e).__name__, "-", e)

**LayerNorm is bit-identical** whether the token arrives alone or in a batch of 8. **BatchNorm gives the same token different outputs** depending on its companions — and with a batch of one it degenerates entirely.

That is the whole argument for transformers using LayerNorm: autoregressive decoding generates **one token at a time**, so there is no meaningful batch, and BatchNorm's train/inference statistics differ. Variable-length sequences and small batches make it worse.

The manual check also confirms the formula: subtract the mean over the token's own features, divide by that token's standard deviation.

---
## Part 4 — The O(n²) wall and the KV cache

Lesson 3's two arithmetic claims:
- Attention compute is $O(n^2 d)$ — **doubling context quadruples it** — while the FFN is only $O(n d^2)$.
- The KV cache is $2 \times L \times H \times d_{head} \times n \times \text{bytes}$ and grows with **every generated token**.

Both are worth computing once so the numbers are yours.

In [ ]:
d_model, n_heads, n_layers = 4096, 32, 32
d_head, d_ff = d_model // n_heads, 4 * d_model

def attn_flops(n):  return 4 * n * n * d_model          # QK^T and (scores)V
def ffn_flops(n):   return 2 * n * d_model * d_ff * 2   # two projections

lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768, 131072]
print(f"{'context':>9} {'attention GFLOPs':>18} {'FFN GFLOPs':>12} {'attn share':>11} {'vs 512':>9}")
base = attn_flops(512)
for n in lengths:
    a, f = attn_flops(n), ffn_flops(n)
    print(f"{n:>9} {a/1e9:>18.1f} {f/1e9:>12.1f} {a/(a+f):>10.0%} {a/base:>9.0f}x")

In [ ]:
def kv_cache_gb(n_tokens, layers=32, heads=32, dh=128, bytes_per=2, kv_heads=None):
    """2 (K and V) x layers x kv_heads x d_head x tokens x bytes."""
    kv_heads = kv_heads if kv_heads is not None else heads
    return 2 * layers * kv_heads * dh * n_tokens * bytes_per / 1e9

print("KV cache for a 7B-class model (32 layers, 32 heads, d_head 128, fp16)\n")
print(f"{'context':>9} {'MHA (32 kv)':>13} {'GQA g=8':>10} {'MQA (1 kv)':>12}")
for n in [1024, 2048, 4096, 8192, 16384, 32768]:
    print(f"{n:>9} {kv_cache_gb(n):>12.2f}G {kv_cache_gb(n, kv_heads=8):>9.2f}G {kv_cache_gb(n, kv_heads=1):>11.2f}G")

print(f"\nAt 8k context, batch of 8 concurrent requests:")
for label, kv in [("MHA", 32), ("GQA g=8", 8), ("MQA", 1)]:
    print(f"  {label:<8} {kv_cache_gb(8192, kv_heads=kv) * 8:>6.1f} GB of cache "
          f"(model weights in fp16 are ~14 GB)")

**Read the two tables together.**

The first shows attention's share of FLOPs climbing with context: at short lengths the FFN dominates, and attention only takes over as $n$ grows past $d_{model}$. The `vs 512` column is the quadratic — each doubling multiplies attention cost by 4.

The second is the number that drives production decisions. At 8k context with a batch of 8, an MHA cache **exceeds the model weights**. GQA with 8 groups cuts it 4×, at close to no quality cost — which is precisely why Llama 2 70B onward use it. And note what GQA does *not* change: query and output projections are untouched, so training compute is essentially unaffected. This is an inference-memory optimization.

---
## Part 5 — Reproduce an fp16 overflow, then fix it with loss scaling

Lesson 4's claim: fp16 has max ≈ 65504, so large values overflow to `inf` and then `NaN`; small gradients underflow to zero. Loss scaling fixes underflow; bf16's wider exponent range fixes overflow.

Watch all three happen.

In [ ]:
fp16_max = torch.finfo(torch.float16).max
fp16_tiny = torch.finfo(torch.float16).tiny
print(f"fp16  max {fp16_max:>12.1f}   smallest normal {fp16_tiny:.2e}")
print(f"bf16  max {torch.finfo(torch.bfloat16).max:>12.3e}   (same exponent range as fp32)")
print(f"fp32  max {torch.finfo(torch.float32).max:.3e}\n")

# --- overflow ---
big = torch.tensor([60000.0, 70000.0], dtype=torch.float32)
print("value 70000 stored as fp16 ->", big.half()[1].item(), " (overflowed)")
print("value 70000 stored as bf16 ->", big.bfloat16()[1].item(), " (fine)\n")

# --- underflow, and loss scaling as the fix ---
tiny_grad = torch.tensor([1e-8], dtype=torch.float32)
print(f"gradient 1e-8 in fp16                      -> {tiny_grad.half().item():.2e}  (underflowed to zero)")
scaled = (tiny_grad * 2**16).half()
print(f"same gradient scaled by 2^16, then unscaled -> {(scaled.float() / 2**16).item():.2e}  (survived)")

In [ ]:
# Fingerprint of a dynamic loss scaler hitting overflow: it halves the scale and skips the step.
scale, history = 2.0**16, []
grad_magnitude = 1.5      # large enough that scale * grad overflows fp16 at first
for step in range(12):
    scaled_grad = torch.tensor([grad_magnitude * scale], dtype=torch.float32).half()
    overflowed = not torch.isfinite(scaled_grad).all()
    history.append((step, scale, overflowed))
    scale = scale / 2 if overflowed else scale        # skip step, halve scale

print(f"{'step':>5} {'loss scale':>12} {'overflow?':>10} {'action':>16}")
for step, sc, ov in history:
    print(f"{step:>5} {sc:>12.0f} {str(ov):>10} {'SKIP + halve' if ov else 'apply update':>16}")
print("\nRepeated 'SKIP + halve' early in training is the fingerprint of fp16 overflow.")

**All three behaviours are now things you have seen.** 70000 becomes `inf` in fp16 but is fine in bf16 (same exponent range as fp32, fewer mantissa bits). A 1e-8 gradient underflows to exactly zero in fp16 — and survives when multiplied by the loss scale and unscaled afterwards.

The last table is the practical diagnostic from Lesson 4: **a dynamic loss scaler repeatedly skipping steps and halving its scale is the fingerprint of fp16 overflow.** If you see that in a training log, you have your answer before touching the learning rate.

---
## What you can now claim

You have *checked* these, so you can state them without hedging:

- Your backprop derivation matches autograd to machine precision — including that $\partial L/\partial W$ is an outer product and that the forward activations must be kept.
- Gradients measurably vanish across 40 plain layers and measurably survive with residual connections.
- LayerNorm is bit-identical regardless of batch composition; BatchNorm is not, and degenerates at batch size 1.
- Attention's quadratic term and the KV-cache arithmetic on a real 7B-class configuration — including that an MHA cache at 8k × batch 8 exceeds the model weights.
- fp16 overflow, underflow, and the loss-scaler fingerprint.

**What you cannot claim:** none of this is a measurement of your own projects. It is general mechanism, demonstrated.

### Next

Return to [Lesson 5 — Mock Round](05_mock_round.md). The two board questions (backprop for an MLP, KV-cache arithmetic) are exactly Parts 1 and 4 of this notebook — you have already done the work.